In [ ]:


import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# === Step 1: Initialization ===
A = 2
iterations = 30
I_0 = 1
v = 1.5
noise_std = 0.05

# Latency function ℓ(p(s)) = a_e * x + b_e
a_e = np.ones(A)
b_e = np.ones(A) * 0.1

# Initial values
prices = np.linspace(0.1, 0.4, A)
alpha =  np.ones(A) * 3.0
beta =  np.ones(A) * 0.3

# History
alpha_hist, beta_hist, prices_hist, flows_hist = [], [], [], []
iteration_logs = []

# === Functions ===
def latency(x):
    return a_e * x + b_e

def compute_I_n(n):
    return int(I_0 * (v ** n))

def compute_delta_n(I_n):
    return I_n ** (-1 / 4)

# === Step : Compute Flows ===
def update_flows(prices):
    utilities = 1 / (1 + prices)
    flows = utilities / np.sum(utilities)
    return flows  

# === Step 3: Parameter Estimation ===
def estimate_parameters(i, observed_lambda, t_values, prev_alpha, prev_beta):
    # === CDL Step 3: Estimating Parameters α and β ===
    def objective(params):
        a_i, b_i = params
        pred = a_i - b_i * t_values
        return np.sum((observed_lambda - pred) ** 2)
    result = minimize(objective, [prev_alpha, prev_beta])
    return result.x

def solve_kkt(alpha_i, beta_i, prices_i, i):
    sum_others = np.sum(prices_i) - prices_i[i]
    p = (alpha_i + beta_i * sum_others) / (2 * beta_i)
    return p  

# === CDL Iteration ===
for n in range(iterations):
    # CDL Step 1: Compute interval length and delta
    I_n = compute_I_n(n)
    delta_n = compute_delta_n(I_n)
    p_hat = np.copy(prices)

    for m in range(A + 1):
        t_values = []
        observed_lambda = []
        
        # Simulate A+1 × Iₙ steps in this stage
        for t in range(I_n):
            current_time = n * (A + 1) * I_n + m * I_n + t
            i = m % A   #（哪個player）

            prices[i] = p_hat[i] + delta_n  
            
            # Flow update
            flows = update_flows(prices)
            t_vec = latency(flows)

            t_values.append(t_vec[i])
            # Simulate lambda observation with noise
            lam = alpha[i] - beta[i] * t_vec[i] + np.random.normal(0, noise_std)
            observed_lambda.append(lam)
            
        # CDL Step 3: Estimate α and β using observed data
        alpha[i], beta[i] = estimate_parameters(i, np.array(observed_lambda), np.array(t_values), alpha[i], beta[i])
        
        # CDL Step 4: Compute new price for player i using KKT
        prices[i] = solve_kkt(alpha[i], beta[i], prices, i)
        
    #  flow update
    flows = update_flows(prices)

    alpha_hist.append(np.copy(alpha))
    beta_hist.append(np.copy(beta))
    prices_hist.append(np.copy(prices))
    flows_hist.append(np.copy(flows))


    iteration_logs.append({
        "Iteration": n + 1,
        "Alpha": np.round(alpha, 4),
        "Beta": np.round(beta, 4),
        "Prices": np.round(prices, 4),
        "Flows": np.round(flows, 4)
    })

# Convert to arrays
alpha_hist = np.array(alpha_hist)
beta_hist = np.array(beta_hist)
prices_hist = np.array(prices_hist)
flows_hist = np.array(flows_hist)

for log in iteration_logs:
    print(f"Iter {log['Iteration']:3d} | Alpha: {log['Alpha']} | Beta: {log['Beta']} | Price: {log['Prices']} | Flow: {log['Flows']}")

# === Visualization ===
fig, axs = plt.subplots(3, 2, figsize=(14, 12))

for i in range(A):
    axs[0, 0].plot(alpha_hist[:, i], label=f'Alpha {i+1}')
axs[0, 0].set_title("Alpha Evolution")
axs[0, 0].set_xlabel("Iteration")
axs[0, 0].set_ylabel("Alpha")
axs[0, 0].legend()
axs[0, 0].grid(True)

for i in range(A):
    axs[0, 1].plot(beta_hist[:, i], label=f'Beta {i+1}')
axs[0, 1].set_title("Beta Evolution")
axs[0, 1].set_xlabel("Iteration")
axs[0, 1].set_ylabel("Beta")
axs[0, 1].legend()
axs[0, 1].grid(True)

for i in range(A):
    axs[1, 0].plot(prices_hist[:, i], label=f'Price {i+1}')
axs[1, 0].set_title("Price Evolution")
axs[1, 0].set_xlabel("Iteration")
axs[1, 0].set_ylabel("Price")
axs[1, 0].legend()
axs[1, 0].grid(True)

for i in range(A):
    axs[1, 1].plot(flows_hist[:, i], label=f'Flow {i+1}')
axs[1, 1].set_title("Flow Evolution")
axs[1, 1].set_xlabel("Iteration")
axs[1, 1].set_ylabel("Flow")
axs[1, 1].legend()
axs[1, 1].grid(True)

for i in range(A):
    axs[2, 0].plot(alpha_hist[:, i], beta_hist[:, i], label=f'Path {i+1}')
axs[2, 0].set_title("Alpha vs Beta")
axs[2, 0].set_xlabel("Alpha")
axs[2, 0].set_ylabel("Beta")
axs[2, 0].legend()
axs[2, 0].grid(True)

for i in range(A):
    axs[2, 1].scatter(prices_hist[:, i], flows_hist[:, i], alpha=0.6, label=f'Path {i+1}')
axs[2, 1].set_title("Price vs Flow")
axs[2, 1].set_xlabel("Price")
axs[2, 1].set_ylabel("Flow")
axs[2, 1].legend()
axs[2, 1].grid(True)

plt.tight_layout()
plt.show()
